In [ ]:
# SPDX-License-Identifier: Apache-2.0


# Fine-Tune Image Model

Run this notebook top to bottom. It trains a small person LoRA with Z-Image and Musubi Tuner on a Brev A100, then generates a short gallery and a downloadable ZIP.

Default path for this workshop:

- download the prepared public Google Drive dataset
- train with trigger `jalen`
- generate a professional headshot, jacket/editorial portrait, action portrait, talk-show shot, campaign image, and thumbnail-style image


## Replacing The Dataset

To train a different person, replace the public Drive folder with your own folder or ZIP shaped like this:

```text
dataset/
  images/
    image_001.jpg
    image_001.txt
    image_002.jpg
    image_002.txt
    ...
  sample_prompts.txt
```

Use at least 12 images. 20-24 is better. Every image needs a same-name `.txt` caption. Put the trigger word first:

```text
my_subject, a photo of a person, close-up headshot, wearing a black jacket
my_subject, a photo of a person, chest-up portrait outdoors, natural lighting
```

Then edit `variables.env`:

```bash
DATASET_URL=https://drive.google.com/drive/folders/...
TRIGGER=my_subject
OUTPUT_NAME=my_subject_zimage
```

Also update `sample_prompts.txt` to use the same trigger.


## 1. Load Settings

This reads `variables.env`, sets the project root, and prints the settings that matter. If you change `variables.env`, rerun this cell.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import textwrap

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)


def load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value


load_env_file(ROOT / "variables.env")

DATASET_URL = os.environ.get("DATASET_URL", "").strip()
TRIGGER = os.environ.get("TRIGGER", "jalen").strip()
OUTPUT_NAME = os.environ.get("OUTPUT_NAME", f"{TRIGGER}_zimage").strip()
TRAIN_RESOLUTION = int(os.environ.get("TRAIN_RESOLUTION", "768"))
MIN_IMAGES = int(os.environ.get("MIN_IMAGES", "12"))
RECOMMENDED_IMAGES = int(os.environ.get("RECOMMENDED_IMAGES", "20"))
MAX_TRAIN_STEPS = os.environ.get("MAX_TRAIN_STEPS", "800").strip()
NETWORK_DIM = int(os.environ.get("NETWORK_DIM", "32"))
NETWORK_ALPHA = int(os.environ.get("NETWORK_ALPHA", str(NETWORK_DIM)))
LEARNING_RATE = os.environ.get("LEARNING_RATE", "1e-4").strip()
INFER_STEPS = int(os.environ.get("INFER_STEPS", "24"))

DATASET_DIR = ROOT / "data" / "dataset"
PROMPT_FILE = Path(os.environ.get("PROMPT_FILE", "data/dataset/sample_prompts.txt"))
if not PROMPT_FILE.is_absolute():
    PROMPT_FILE = ROOT / PROMPT_FILE

print(f"Repo root: {ROOT}")
print(f"Drive dataset: {DATASET_URL or '(missing)'}")
print(f"Trigger: {TRIGGER}")
print(f"Output name: {OUTPUT_NAME}")
print(f"Training steps: {MAX_TRAIN_STEPS}")
print(f"LoRA dim/alpha: {NETWORK_DIM}/{NETWORK_ALPHA}")
print(f"Prompt file: {PROMPT_FILE}")

if not DATASET_URL:
    raise ValueError("DATASET_URL is empty. Add a public Google Drive folder or ZIP URL to variables.env.")


## 2. Check GPU

A100 is the intended workshop target. If this does not show an NVIDIA GPU, stop and fix the Brev resource before training.


In [ ]:
subprocess.run(["nvidia-smi"], check=True)


## 3. Setup Musubi Tuner

The Brev Launchable post-build should already do this. This cell only runs setup again if Musubi is missing.


In [ ]:
musubi_train = ROOT / "vendor" / "musubi-tuner" / "src" / "musubi_tuner" / "zimage_train_network.py"
accelerate_config = ROOT / "configs" / "accelerate-single-gpu.yaml"

if musubi_train.exists() and accelerate_config.exists():
    print("Musubi and Accelerate config already present.")
else:
    subprocess.run(["bash", "scripts/setup_brev.sh"], check=True)


## 4. Download Z-Image

The model is large. The notebook skips this if the weights are already present in `models/z-image`.


In [ ]:
def has_zimage_model() -> bool:
    model_dir = ROOT / "models" / "z-image"
    return (
        any((model_dir / "transformer").glob("*.safetensors"))
        and any((model_dir / "vae").glob("*.safetensors"))
        and any((model_dir / "text_encoder").glob("*.safetensors"))
    )

if has_zimage_model():
    print("Z-Image model files already present.")
else:
    subprocess.run(["bash", "scripts/download_models.sh"], check=True)


## 5. Download Dataset From Google Drive

This pulls the public Drive folder into `data/dataset/` and rewrites `dataset.toml` with Brev-local paths.


In [ ]:
subprocess.run([
    sys.executable,
    "scripts/download_workshop_dataset.py",
    "--dataset", "custom",
    "--url", DATASET_URL,
    "--output", str(DATASET_DIR),
    "--force",
    "--resolution", str(TRAIN_RESOLUTION),
], check=True)

print("Dataset files:")
for path in sorted((DATASET_DIR / "images").iterdir())[:12]:
    print(" ", path.relative_to(ROOT))


## 6. Validate Images And Captions

This catches the common mistakes before wasting GPU time: too few images, missing `.txt` captions, empty captions, or captions missing the trigger word.


In [ ]:
subprocess.run([
    sys.executable,
    "scripts/validate_dataset.py",
    "--dataset", str(DATASET_DIR),
    "--trigger", TRIGGER,
    "--min-images", str(MIN_IMAGES),
    "--recommended-images", str(RECOMMENDED_IMAGES),
], check=True)


## 7. Review A Few Training Examples

You should see one person per image and captions that start with the trigger word.


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

image_paths = sorted(
    p for p in (DATASET_DIR / "images").iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
)
preview = image_paths[:8]
cols = 4
rows = (len(preview) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4.7 * rows))
axes = [axes] if len(preview) == 1 else list(getattr(axes, "flat", axes))

for ax, image_path in zip(axes, preview):
    ax.imshow(Image.open(image_path).convert("RGB"))
    caption_path = image_path.with_suffix(".txt")
    caption = caption_path.read_text(encoding="utf-8").strip() if caption_path.exists() else "missing caption"
    ax.set_title(textwrap.shorten(caption, width=72), fontsize=9)
    ax.axis("off")
for ax in axes[len(preview):]:
    ax.axis("off")
plt.tight_layout()


## 8. Prepare Sample Prompts

If your Drive dataset includes `sample_prompts.txt`, the notebook uses it. Otherwise it creates a small default prompt file using the current trigger.


In [ ]:
if not PROMPT_FILE.exists():
    PROMPT_FILE.parent.mkdir(parents=True, exist_ok=True)
    PROMPT_FILE.write_text("\n".join([
        f"{TRIGGER}, a photo of a person, modern professional headshot, clean background, natural expression, soft studio lighting, sharp focus --w 768 --h 768 --s {INFER_STEPS} --d 3301 --l 1",
        f"{TRIGGER}, a photo of a person, wearing a black leather jacket, New York rooftop at dusk, editorial magazine portrait, cinematic lighting --w 768 --h 768 --s {INFER_STEPS} --d 3302 --l 1",
        f"{TRIGGER}, a photo of a person, action movie lead, tactical black jacket, dramatic studio lighting, intense expression, poster-quality portrait --w 768 --h 768 --s {INFER_STEPS} --d 3303 --l 1",
        f"{TRIGGER}, a photo of a person, seated on a late-night talk show set, black outfit, warm stage lighting, candid smile --w 768 --h 768 --s {INFER_STEPS} --d 3304 --l 1",
        f"{TRIGGER}, a photo of a person, wearing a premium sportswear launch jacket, urban campaign photography, clean product ad composition, room for headline text --w 768 --h 768 --s {INFER_STEPS} --d 3305 --l 1",
    ]) + "\n", encoding="utf-8")

print(PROMPT_FILE.read_text(encoding="utf-8"))


## 9. Train The LoRA

This is the main GPU step. On A100, the workshop target is a short run rather than perfect convergence.


In [ ]:
train_env = os.environ.copy()
train_env.update({
    "MAX_TRAIN_STEPS": MAX_TRAIN_STEPS,
    "NETWORK_DIM": str(NETWORK_DIM),
    "NETWORK_ALPHA": str(NETWORK_ALPHA),
    "LEARNING_RATE": LEARNING_RATE,
    "OUTPUT_NAME": OUTPUT_NAME,
})

subprocess.run([
    "bash",
    "scripts/train_zimage_lora.sh",
    str(DATASET_DIR / "dataset.toml"),
    str(ROOT / "outputs" / "lora"),
], check=True, env=train_env)


## 10. Pick The Latest Checkpoint

The training script creates a `latest` symlink. If that fails for any reason, this cell falls back to the newest matching checkpoint file.


In [ ]:
lora_dir = ROOT / "outputs" / "lora"
latest_symlink = lora_dir / f"{OUTPUT_NAME}-latest.safetensors"
lora_candidates = sorted(lora_dir.glob(f"{OUTPUT_NAME}*.safetensors"), key=lambda p: p.stat().st_mtime)

if latest_symlink.is_symlink() or latest_symlink.exists():
    LORA_PATH = latest_symlink
elif lora_candidates:
    LORA_PATH = lora_candidates[-1]
else:
    raise FileNotFoundError(f"No LoRA checkpoint found for {OUTPUT_NAME} in {lora_dir}")

print("Using LoRA:", LORA_PATH)


## 11. Generate Images

This uses the trained LoRA and the sample prompts from the dataset.


In [ ]:
gallery_dir = ROOT / "outputs" / "gallery"
gallery_dir.mkdir(parents=True, exist_ok=True)

gen_env = os.environ.copy()
gen_env.update({
    "INFER_STEPS": str(INFER_STEPS),
    "IMAGE_HEIGHT": str(TRAIN_RESOLUTION),
    "IMAGE_WIDTH": str(TRAIN_RESOLUTION),
})

subprocess.run([
    "bash",
    "scripts/generate_zimage_gallery.sh",
    str(LORA_PATH),
    str(PROMPT_FILE),
    str(gallery_dir),
], check=True, env=gen_env)

print("Gallery:", gallery_dir)


## 12. Review Generated Images


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

images = sorted(gallery_dir.glob("*.png")) + sorted(gallery_dir.glob("*.jpg")) + sorted(gallery_dir.glob("*.jpeg"))
print(f"Found {len(images)} images")

if images:
    cols = min(3, len(images))
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
    axes = [axes] if len(images) == 1 else list(getattr(axes, "flat", axes))
    for ax, path in zip(axes, images):
        ax.imshow(Image.open(path).convert("RGB"))
        ax.set_title(path.name, fontsize=9)
        ax.axis("off")
    for ax in axes[len(images):]:
        ax.axis("off")
    plt.tight_layout()
else:
    print("No images found yet. Check the generation cell output above.")


## 13. Make Your Own Prompt

Normally, every prompt needs the LoRA trigger word so the model knows which learned subject to use. This cell adds the trigger automatically, so type only the scene you want.

Example inputs:

```text
professional headshot in a navy suit, clean studio background
wearing a black leather jacket on a New York rooftop at dusk
vacation photo on a sunny Hawaii beach, candid smile
```

If you already include the trigger yourself, the cell will not add it twice.


In [ ]:
import random
from pathlib import Path
from IPython.display import clear_output, display
import ipywidgets as widgets
from PIL import Image

custom_prompt_dir = ROOT / "outputs" / "custom_prompts"
custom_prompt_dir.mkdir(parents=True, exist_ok=True)
gallery_dir = ROOT / "outputs" / "gallery"
gallery_dir.mkdir(parents=True, exist_ok=True)

prompt_box = widgets.Textarea(
    value="professional headshot in a navy suit, clean studio background, soft studio lighting",
    placeholder="Describe the image you want. The trigger is added automatically.",
    description="Prompt",
    layout=widgets.Layout(width="100%", height="110px"),
)
seed_box = widgets.IntText(
    value=random.randint(10000, 99999),
    description="Seed",
    layout=widgets.Layout(width="220px"),
)
generate_button = widgets.Button(
    description="Generate image",
    button_style="success",
    icon="image",
)
custom_output = widgets.Output()


def prompt_with_trigger(text: str, seed: int) -> str:
    cleaned = " ".join(text.strip().split())
    if not cleaned:
        raise ValueError("Type a prompt first.")

    trigger_prefix = TRIGGER.lower() + ","
    if cleaned.lower().startswith(trigger_prefix) or cleaned.lower() == TRIGGER.lower():
        prompt = cleaned
    else:
        prompt = f"{TRIGGER}, a photo of a person, {cleaned}"

    if "--w" not in prompt:
        prompt = f"{prompt} --w {TRAIN_RESOLUTION} --h {TRAIN_RESOLUTION} --s {INFER_STEPS} --d {seed} --l 1"
    return prompt


def on_generate_clicked(_):
    with custom_output:
        clear_output()
        seed = int(seed_box.value)
        prompt = prompt_with_trigger(prompt_box.value, seed)
        prompt_file = custom_prompt_dir / f"custom_prompt_{seed}.txt"
        prompt_file.write_text(prompt + "\n", encoding="utf-8")

        print("Generated prompt:")
        print(prompt)
        print()

        before = set(gallery_dir.glob("*.png")) | set(gallery_dir.glob("*.jpg")) | set(gallery_dir.glob("*.jpeg"))
        subprocess.run([
            "bash",
            "scripts/generate_zimage_gallery.sh",
            str(LORA_PATH),
            str(prompt_file),
            str(gallery_dir),
        ], check=True)
        after = set(gallery_dir.glob("*.png")) | set(gallery_dir.glob("*.jpg")) | set(gallery_dir.glob("*.jpeg"))
        new_images = sorted(after - before, key=lambda path: path.stat().st_mtime)
        if not new_images:
            new_images = sorted(after, key=lambda path: path.stat().st_mtime)[-1:]

        for image_path in new_images:
            print("Saved:", image_path.relative_to(ROOT))
            display(Image.open(image_path).convert("RGB"))

        seed_box.value = random.randint(10000, 99999)


generate_button.on_click(on_generate_clicked)
display(prompt_box, widgets.HBox([seed_box, generate_button]), custom_output)


## 14. Download Checkpoints And Images

This creates a ZIP with the LoRA checkpoint, prompt file, generated images, and dataset captions. The button downloads it through Jupyter.


In [ ]:
from datetime import datetime
from html import escape
from pathlib import Path
from urllib.parse import quote
import json
import zipfile
from IPython.display import HTML, display

outputs_dir = ROOT / "outputs"
download_dir = outputs_dir / "downloads"
download_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
bundle_path = download_dir / f"{OUTPUT_NAME}_checkpoints_images_prompts_{timestamp}.zip"

lora_files = sorted(p for p in (outputs_dir / "lora").glob(f"{OUTPUT_NAME}*.safetensors") if not p.is_symlink())
gallery_files = []
for pattern in ("**/*.png", "**/*.jpg", "**/*.jpeg", "**/*.webp"):
    gallery_files.extend(sorted((outputs_dir / "gallery").glob(pattern)))
caption_files = sorted((DATASET_DIR / "images").glob("*.txt"))
custom_prompt_files = sorted((outputs_dir / "custom_prompts").glob("*.txt"))
extra_files = [PROMPT_FILE, *custom_prompt_files, ROOT / "variables.env", DATASET_DIR / "dataset.toml"]

files_to_zip = []
for file_path in [*lora_files, *gallery_files, *caption_files, *extra_files]:
    if file_path.exists() and file_path.is_file() and file_path not in files_to_zip:
        files_to_zip.append(file_path)

manifest = [
    "NY Tech Week fine-tuning workshop outputs",
    f"created_at={timestamp}",
    f"trigger={TRIGGER}",
    f"output_name={OUTPUT_NAME}",
    f"lora_files={len(lora_files)}",
    f"gallery_images={len(gallery_files)}",
    f"caption_files={len(caption_files)}",
    "",
    "Files:",
]

with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_STORED) as zf:
    for file_path in files_to_zip:
        arcname = file_path.relative_to(ROOT).as_posix()
        zf.write(file_path, arcname)
        manifest.append(arcname)
    zf.writestr("MANIFEST.txt", "\n".join(manifest) + "\n")

size_mb = bundle_path.stat().st_size / (1024 * 1024)
relative_href = quote(bundle_path.relative_to(ROOT).as_posix())
button_label = f"Download checkpoint + images ({size_mb:.1f} MB)"

download_js = (
    "const path = window.location.pathname;"
    "const base = path.includes('/lab/') ? path.split('/lab/')[0] : "
    "(path.includes('/notebooks/') ? path.split('/notebooks/')[0] : '');"
    f"window.location.href = base + '/files/{relative_href}?download=1';"
    "return false;"
)

display(HTML(f"""
<div style="border: 1px solid #d6d6d6; padding: 16px; border-radius: 6px; max-width: 720px;">
  <div style="font-weight: 700; margin-bottom: 8px;">Output bundle ready</div>
  <div style="margin-bottom: 12px; color: #444;">
    {len(lora_files)} checkpoint file(s), {len(gallery_files)} generated image(s), {len(caption_files)} caption file(s).
  </div>
  <a href="#" onclick={json.dumps(download_js)} style="display: inline-block; background: #76b900; color: #111; padding: 10px 14px; border-radius: 4px; text-decoration: none; font-weight: 700;">
    {escape(button_label)}
  </a>
  <div style="margin-top: 10px; font-size: 12px; color: #666;">
    Saved in the Brev workspace as <code>{escape(bundle_path.relative_to(ROOT).as_posix())}</code>.
  </div>
</div>
"""))

print("Created output bundle:", bundle_path)


## Optional Cleanup

Run this only after downloading outputs. It removes the downloaded/prepared dataset from the Brev workspace.


In [ ]:
# import shutil
# shutil.rmtree(DATASET_DIR, ignore_errors=True)
# print("Deleted prepared dataset:", DATASET_DIR)
